In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 19:32:48.850737: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 19:32:49.562526: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

2023-07-02 19:32:51,373 [DEBUG] [Rain] Rain is initialized
2023-07-02 19:32:51,374 [DEBUG] [Provisioner] Creating coordinator
2023-07-02 19:32:51,375 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-02 19:32:51,377 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_async()

2023-07-02 19:32:51,383 [DEBUG] [Rain] Creating workers
2023-07-02 19:32:51,390 [INFO] [Provisioner] provisioner is serving
2023-07-02 19:32:51,390 [DEBUG] [Provisioner] Starting coordinator
2023-07-02 19:32:51,392 [INFO] [Coordinator] coordinator is serving
2023-07-02 19:32:51,392 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-02 19:32:51,397 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 19:32:51,398 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-02 19:32:51,399 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-02 19:32:51,402 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-02 19:32:51,404 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-02 19:32:51,406 [INFO] [Worker_50153] Worker is running on port: 50153
2023-07-02 19:32:51,406 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 10ms/step - loss: 0.6965 - accuracy: 0.7803
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.6959 - accuracy: 0.7822
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.6940 - accuracy: 0.7799
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.3068 - accuracy: 0.9075
sending data to coordinator
157/157 [==============================] - 2s 10ms/step - loss: 0.3037 - accuracy: 0.9083
sending data to coordinator


2023-07-02 19:33:28,132 [DEBUG] [DividerAmbassador] divider received: Executed! for worker2
2023-07-02 19:33:28,133 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/2_2_trained.pkl from worker2
2023-07-02 19:33:28,147 [DEBUG] [DividerAmbassador] divider received: Executed! for worker1
2023-07-02 19:33:28,148 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/1_1_trained.pkl from worker1
2023-07-02 19:33:28,298 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/2_2_trained.pkl in divider
2023-07-02 19:33:28,316 [DEBUG] [Divider] update is done by worker 2
2023-07-02 19:33:28,332 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/1_1_trained.pkl in divider
2023-07-02 19:33:28,349 [DEBUG] [Divider] update is done by worker 1
2023-07-02 19:33:28,355 [DEBUG] [Divider] Sending file: ../../../Divider/divider/data/2.pkl
2023-07-02 19:33:28,358 [DEBUG] [DividerAmbassador] divider is sending

sending data to coordinator


2023-07-02 19:33:29,880 [DEBUG] [DividerAmbassador] divider received: Executed! for worker3
2023-07-02 19:33:29,882 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/3_3_trained.pkl from worker3
2023-07-02 19:33:30,223 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/3_3_trained.pkl in divider
2023-07-02 19:33:30,235 [DEBUG] [Divider] update is done by worker 3
2023-07-02 19:33:30,281 [DEBUG] [Divider] Sending file: ../../../Divider/divider/data/3.pkl
2023-07-02 19:33:30,283 [DEBUG] [DividerAmbassador] divider is sending information file to the coordinator
2023-07-02 19:33:30,565 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-02 19:33:30,567 [DEBUG] [Divider] Iteration 1/3 complete for worker 3.
2023-07-02 19:33:30,568 [DEBUG] [Divider] Starting iteration 2/3
2023-07-02 19:33:30,569 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-02 19:33:42,984 [DEBUG] [DividerAmbassador] divi

...

2023-07-02 19:34:02,642 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/1_1_trained.pkl in divider


103/157 [==================>...........] - ETA: 0s - loss: 0.1687 - accuracy: 0.9496

2023-07-02 19:34:02,664 [DEBUG] [Divider] update is done by worker 1
2023-07-02 19:34:02,715 [DEBUG] [Divider] Sending file: ../../../Divider/divider/data/1.pkl
2023-07-02 19:34:02,718 [DEBUG] [DividerAmbassador] divider is sending information file to the coordinator
2023-07-02 19:34:02,857 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-02 19:34:02,859 [DEBUG] [Divider] Iteration 3/3 complete for worker 1.


157/157 [==============================] - 1s 7ms/step - loss: 0.1605 - accuracy: 0.9505
sending data to coordinator
157/157 [==============================] - 1s 6ms/step - loss: 0.1665 - accuracy: 0.9501
sending data to coordinator


2023-07-02 19:34:04,064 [DEBUG] [DividerAmbassador] divider received: Executed! for worker3
2023-07-02 19:34:04,066 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/3_3_trained.pkl from worker3
2023-07-02 19:34:04,161 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/3_3_trained.pkl in divider
2023-07-02 19:34:04,172 [DEBUG] [Divider] update is done by worker 3
2023-07-02 19:34:04,173 [DEBUG] [DividerAmbassador] divider received: Executed! for worker2
2023-07-02 19:34:04,176 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/2_2_trained.pkl from worker2
2023-07-02 19:34:04,242 [DEBUG] [Divider] Sending file: ../../../Divider/divider/data/3.pkl
2023-07-02 19:34:04,244 [DEBUG] [DividerAmbassador] divider is sending information file to the coordinator
2023-07-02 19:34:04,455 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/2_2_trained.pkl in divider
2023-07-02 19:34:04,465 [DEBUG] 

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1097 - accuracy: 0.9678

Test accuracy: 96.8%


In [11]:
model = rain.train_centralized_sync()

2023-07-02 19:34:04,926 [DEBUG] [Rain] Creating workers
2023-07-02 19:34:04,929 [INFO] [Provisioner] provisioner is serving
2023-07-02 19:34:04,929 [DEBUG] [Provisioner] Starting coordinator
2023-07-02 19:34:04,931 [INFO] [Coordinator] coordinator is serving
2023-07-02 19:34:04,931 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-02 19:34:04,933 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 19:34:04,934 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-02 19:34:04,935 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-02 19:34:04,937 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-02 19:34:04,937 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-02 19:34:04,939 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-02 19:34:04,939 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-02 19:34:04,942 [I

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1556 - accuracy: 0.9549
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1574 - accuracy: 0.9528
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1577 - accuracy: 0.9538
Epoch 2/2
157/157 [==============================] - 1s 9ms/step - loss: 0.1349 - accuracy: 0.9594
sending data to coordinator
157/157 [==============================] - 1s 9ms/step - loss: 0.1336 - accuracy: 0.9588
sending data to coordinator
sending data to coordinator


2023-07-02 19:34:37,841 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:34:37,857 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:34:37,859 [INFO] [Coordinator] thread 1 is done
2023-07-02 19:34:37,895 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:34:37,938 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/1_1_trained.pkl in coordinator
2023-07-02 19:34:37,939 [INFO] [Coordinator] thread 2 is done
2023-07-02 19:34:38,013 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/2_1_trained.pkl in coordinator
2023-07-02 19:34:38,014 [INFO] [Coordinator] thread 3 is done
2023-07-02 19:34:38,102 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/3_1_trained.pkl in coordinator
2023-07-02 19:34:38,181 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023-07-02 19:34:38,254 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023

Epoch 1/2
Epoch 1/2
Epoch 1/2


2023-07-02 19:34:55.384075: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 62720000 exceeds 10% of free system memory.


157/157 [==============================] - 2s 7ms/step - loss: 0.1308 - accuracy: 0.9620
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1375 - accuracy: 0.9586
Epoch 2/2
157/157 [==============================] - 2s 9ms/step - loss: 0.1268 - accuracy: 0.9624
Epoch 2/2
157/157 [==============================] - 1s 9ms/step - loss: 0.1174 - accuracy: 0.9640
sending data to coordinator
157/157 [==============================] - 1s 8ms/step - loss: 0.1200 - accuracy: 0.9632
sending data to coordinator
157/157 [==============================] - 1s 7ms/step - loss: 0.1131 - accuracy: 0.9661
sending data to coordinator


2023-07-02 19:34:59,547 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:34:59,549 [INFO] [Coordinator] thread 1 is done
2023-07-02 19:34:59,626 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:34:59,635 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/1_2_trained.pkl in coordinator
2023-07-02 19:34:59,636 [INFO] [Coordinator] thread 2 is done
2023-07-02 19:34:59,716 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/2_2_trained.pkl in coordinator
2023-07-02 19:34:59,734 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:34:59,735 [INFO] [Coordinator] thread 3 is done
2023-07-02 19:34:59,811 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/3_2_trained.pkl in coordinator
2023-07-02 19:34:59,882 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023-07-02 19:34:59,954 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1270 - accuracy: 0.9620
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1131 - accuracy: 0.9661
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1050 - accuracy: 0.9694
sending data to coordinator
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.0992 - accuracy: 0.9706
sending data to coordinator


2023-07-02 19:35:18,612 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:35:18,621 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:35:18,622 [INFO] [Coordinator] thread 1 is done
2023-07-02 19:35:18,635 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 19:35:18,695 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/1_3_trained.pkl in coordinator
2023-07-02 19:35:18,695 [INFO] [Coordinator] thread 2 is done
2023-07-02 19:35:18,768 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/2_3_trained.pkl in coordinator
2023-07-02 19:35:18,769 [INFO] [Coordinator] thread 3 is done
2023-07-02 19:35:18,838 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/3_3_trained.pkl in coordinator
2023-07-02 19:35:18,905 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023-07-02 19:35:18,973 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023

In [12]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0752 - accuracy: 0.9772

Test accuracy: 97.7%
